Messages

Messages are the fundamental unit of context for models in LangChain. They represent the input and output of models, carrying both the content and metadata needed to represent the state of a conversation when interacting with an LLM. Messages are objects that contain:

    Role - Identifies the message type (e.g. system, user)
    Content - Represents the actual content of the message (like text, images, audio, documents, etc.)
    Metadata - Optional fields such as response information, message IDs, and token usage

LangChain provides a standard message type that works across all model providers, ensuring consistent behavior regardless of the model being called.

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain.chat_models import init_chat_model
model = init_chat_model("groq:openai/gpt-oss-120b")

In [3]:
model.invoke("Please tell what is artificial intelligence short and crisp")

AIMessage(content='Artificial intelligence (AI) is the field of computer science that creates systems capable of performing tasks that normally require human intelligence—such as learning, reasoning, perception, and decision‑making—by processing data, recognizing patterns, and adapting to new information.', additional_kwargs={'reasoning_content': 'The user asks: "Please tell what is artificial intelligence short and crisp". So we need to give a brief definition. No disallowed content. So we comply.'}, response_metadata={'token_usage': {'completion_tokens': 93, 'prompt_tokens': 80, 'total_tokens': 173, 'completion_time': 0.192553092, 'completion_tokens_details': {'reasoning_tokens': 34}, 'prompt_time': 0.003092354, 'prompt_tokens_details': None, 'queue_time': 0.308434704, 'total_time': 0.195645446}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_87b3c396db', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a


Message Prompts

Alternatively, you can pass in a list of messages to the model by providing a list of message objects.

Message types

    System message - Tells the model how to behave and provide context for interactions
    Human message - Represents user input and interactions with the model
    AI message - Responses generated by the model, including text content, tool calls, and metadata
    Tool message - Represents the outputs of tool calls

System Message

A SystemMessage represent an initial set of instructions that primes the model’s behavior. You can use a system message to set the tone, define the model’s role, and establish guidelines for responses.
Human Message

A HumanMessage represents user input and interactions. They can contain text, images, audio, files, and any other amount of multimodal content.
AI Message

An AIMessage represents the output of a model invocation. They can include multimodal data, tool calls, and provider-specific metadata that you can later access.
Tool Message

For models that support tool calling, AI messages can contain tool calls. Tool messages are used to pass the results of a single tool execution back to the model.


In [4]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage

messages = [
    SystemMessage(content="You are a helpful assistant."), 
    HumanMessage(content="Please tell what is artificial intelligence short and crisp")
]

response = model.invoke(messages)
response.content

'**Artificial Intelligence (AI)** – the field of computer science that creates systems capable of performing tasks that normally require human intelligence, such as learning, reasoning, perception, and decision‑making.'

In [5]:
system_msg = SystemMessage("""
You are a senior Python developer with expertise in web frameworks.
Always provide code examples and explain your reasoning.
Be concise but thorough in your explanations.
""")

messages = [
    system_msg,
    HumanMessage("How do I create a REST API?")
]
response = model.invoke(messages)
print(response.content)

## TL;DR  
Pick a Python web framework, define **resources** (the data you expose), map HTTP verbs to CRUD operations, serialize/validate payloads, and run the app behind a WSGI/ASGI server.  
Below is a **complete, minimal FastAPI project** (the most popular modern choice) that you can copy‑paste, run, and extend.

---

## 1. Choose a framework  

| Framework | When to use | Pros | Cons |
|-----------|-------------|------|------|
| **FastAPI** | New projects, async, automatic OpenAPI docs | Very fast, type‑hints → validation, built‑in docs, async ready | Slightly steeper learning curve if you’ve never used Pydantic |
| **Flask** | Tiny services, already familiar | Minimalist, huge ecosystem | You have to add validation, docs, etc. yourself |
| **Django REST Framework** | Large monoliths, existing Django apps | Batteries‑included (auth, pagination, browsable API) | Heavier, tied to Django ORM |
| **Falcon / Bottle / aiohttp** | Very low‑level control | Minimal overhead | You’ll write m

In [17]:
from langchain.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage

messages = [
    SystemMessage(content="You are a helpful assistant."), 
    HumanMessage(content="Please tell what is artificial intelligence short and crisp"),
    AIMessage(content="Artificial Intelligence (AI) is the simulation of human intelligence in machines that are programmed to think, learn, and make decisions like humans.")
]

response = model.invoke(messages)
response.content

'Artificial Intelligence (AI) is the simulation of human intelligence by computers—enabling machines to learn, reason, and make decisions.'

In [10]:
human_msg = HumanMessage(
    content="Please tell what is artificial intelligence short and crisp",
    name="user1",
    id="msg1",
)
print(human_msg)  

content='Please tell what is artificial intelligence short and crisp' additional_kwargs={} response_metadata={} name='user1' id='msg1'


In [12]:
response = model.invoke([
    human_msg,
])
response

AIMessage(content='Artificial intelligence (AI) is the branch of computer science that creates systems capable of performing tasks that normally require human intelligence—such as learning, reasoning, perception, and decision‑making—by processing data, recognizing patterns, and adapting their behavior.', additional_kwargs={'reasoning_content': 'The user wants a short and crisp definition of artificial intelligence. Provide concise answer.'}, response_metadata={'token_usage': {'completion_tokens': 75, 'prompt_tokens': 80, 'total_tokens': 155, 'completion_time': 0.159639465, 'completion_tokens_details': {'reasoning_tokens': 17}, 'prompt_time': 0.003356726, 'prompt_tokens_details': None, 'queue_time': 0.388241188, 'total_time': 0.162996191}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_803c0ba83d', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a13a5d-fd1c-7cc2-9a3b-8d121fbd794b-0', tool_calls=[], invalid_

In [14]:
ai_msg = AIMessage(
    "i'd be happy to help you with that question."
)
messages = [
    SystemMessage("You are a helpful assistant."),
    HumanMessage("Please tell what is artificial intelligence short and crisp"),
    ai_msg,
    HumanMessage("Can you provide a code example?")
]

response = model.invoke(messages)
print(response.content)

**Artificial Intelligence (AI) – short & crisp**  
AI is the field of computer science that builds systems capable of performing tasks that normally require human intelligence—such as learning, reasoning, perception, and decision‑making.

---

## Simple AI Code Example (Python)

Below is a minimal, end‑to‑end example that trains a tiny neural network to recognize handwritten digits (MNIST) using **Keras** (TensorFlow backend). It demonstrates core AI steps: data loading, model definition, training, and inference.

```python
# -------------------------------------------------
# Tiny MNIST classifier – 20 lines of pure Python
# -------------------------------------------------
import tensorflow as tf
from tensorflow.keras import layers, models, datasets

# 1️⃣ Load and preprocess data
(x_train, y_train), (x_test, y_test) = datasets.mnist.load_data()
x_train = x_train[..., tf.newaxis] / 255.0   # shape: (60000,28,28,1)
x_test  = x_test[..., tf.newaxis]  / 255.0

# 2️⃣ Build a simple CNN m

In [15]:
response.usage_metadata

{'input_tokens': 117,
 'output_tokens': 809,
 'total_tokens': 926,
 'output_token_details': {'reasoning': 96}}

In [18]:
ai_message = AIMessage(
    content=[],
    tool_calls=[{
        "name": "get_weather",
        "args": {"location": "San Francisco"},
        "id": "call_123"
    }]
)


weather_result = "Sunny, 72°F"
tool_message = ToolMessage(
    content=weather_result,
    tool_call_id="call_123"  
)

messages = [
    HumanMessage("What's the weather in San Francisco?"),
    ai_message,  
    tool_message, 
]
response = model.invoke(messages)

In [19]:
response

AIMessage(content='The current weather in San\u202fFrancisco is sunny with a temperature of about **72\u202f°F**.', additional_kwargs={'reasoning_content': 'The user asks "What\'s the weather in San Francisco?" The assistant can call get_weather function. We have done that: we responded with function call? Actually we need to output a function call. The prior assistant message was a function call? The assistant used get_weather with location "San Francisco". Then we need to output the function result. The tool responded with "Sunny, 72°F". Now we need to answer the user with that info in a natural language. Probably just say the current weather: "It\'s sunny and 72°F".'}, response_metadata={'token_usage': {'completion_tokens': 140, 'prompt_tokens': 109, 'total_tokens': 249, 'completion_time': 0.292927005, 'completion_tokens_details': {'reasoning_tokens': 110}, 'prompt_time': 0.027582012, 'prompt_tokens_details': None, 'queue_time': 0.315752832, 'total_time': 0.320509017}, 'model_name':